In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

# Load environment variables
load_dotenv(
    r"C:\Users\shekh\OneDrive\Desktop\Tutorial Agentic AI\.env",
    override=True
)

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in your .env file.")

# Initialize the LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    max_retries=0
)

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [2]:
with open("blood_work.txt", "r", encoding="utf-8") as f:
    blood_report = f.read()

print("=== SAMPLE BLOOD REPORT ===")
print(blood_report[:500])

=== SAMPLE BLOOD REPORT ===
Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5–17.5)
Hematocrit:        44%              (Normal: 41–53%)
WBC:               6.8 x10^3/uL     (Normal: 4.5–11.0)
Platelets:         220 x10^3/uL     (Normal: 150–400)

LIPID PANEL
-----------
Total Cholesterol: 238 mg/dL        (Normal: <200)
LDL Cholesterol:   162 mg/dL        (Normal: <100)
HDL Cholesterol:   36 mg/dL         (Normal: 


In [3]:
extraction_prompt = f"""
You are a medical data extraction assistant.

Read the blood report below and extract ALL test values.

For each test:
1. Extract the test name and measured value.
2. Identify the reference range provided in the report.
3. Classify the result as HIGH, LOW, or NORMAL based on that range.
4. If the reference range is missing or the result cannot be classified,
   mark it as UNDETERMINED.
5. Do not invent missing values or reference ranges.

Format your response as:

- Test Name: value | Status: HIGH/LOW/NORMAL/UNDETERMINED
  | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)

extracted_values = extraction_response.content

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
- Hemoglobin: 15.1 g/dL | Status: NORMAL  
  | Reference: 13.5–17.5  

- Hematocrit: 44 % | Status: NORMAL  
  | Reference: 41–53%  

- WBC: 6.8 x10³/µL | Status: NORMAL  
  | Reference: 4.5–11.0  

- Platelets: 220 x10³/µL | Status: NORMAL  
  | Reference: 150–400  

- Total Cholesterol: 238 mg/dL | Status: HIGH  
  | Reference: <200  

- LDL Cholesterol: 162 mg/dL | Status: HIGH  
  | Reference: <100  

- HDL Cholesterol: 36 mg/dL | Status: LOW  
  | Reference: >40  

- Triglycerides: 188 mg/dL | Status: HIGH  
  | Reference: <150  

- Glucose (Fasting): 92 mg/dL | Status: NORMAL  
  | Reference: 70–99  

- HbA1c: 5.3 % | Status: NORMAL  
  | Reference: <5.7%  

- Creatinine: 1.0 mg/dL | Status: NORMAL  
  | Reference: 0.7–1.3  

- eGFR: 82 mL/min | Status: NORMAL  
  | Reference: >60  

- ALT: 28 U/L | Status: NORMAL  
  | Reference: 7–40  

- AST: 25 U/L | Status: NORMAL  
  | Reference: 10–40  

- Bilirubin Total: 0.8 mg/dL | Status: NORMAL  
  | 

In [4]:
diet_prompt = f"""
You are a health information assistant with knowledge of Indian dietary habits.

Based ONLY on the blood work analysis below, provide:

1. HEALTH SUMMARY
   - Write 4–5 short lines explaining the results in simple language.
   - Mention the important abnormal values and their possible significance.
   - Do not diagnose a disease from blood results alone.
   - If the information is insufficient, explicitly say so.

2. INDIAN DIET PLAN

   Include ONLY these two sections:

   (1) Foods to avoid
   (2) Foods to eat more of

   - Suggest practical Indian foods and meals.
   - Make suggestions relevant to the reported abnormal values.
   - Do not recommend supplements or medication.
   - Do not invent abnormalities that are not present in the analysis.
   - If no meaningful dietary recommendation can be made, explain this briefly.

Important:
- Blood test interpretation depends on factors such as age, sex,
  medical history, medications, and laboratory reference ranges.
- Do not present the summary as a medical diagnosis.
- Recommend discussing significant abnormalities with a qualified clinician.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.content)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**HEALTH SUMMARY**  
- Your red‑blood‑cell counts (Hb, Hct, WBC, platelets) are all within normal limits, indicating no obvious anemia or infection.  
- Total cholesterol (238 mg/dL) and LDL‑cholesterol (162 mg/dL) are elevated, which can increase the long‑term risk of atherosclerosis.  
- HDL‑cholesterol is low (36 mg/dL), reducing the protective “good‑cholesterol” effect.  
- Triglycerides are mildly high (188 mg/dL), another factor that can contribute to cardiovascular risk.  
- Blood‑sugar, kidney (creatinine, eGFR) and liver (ALT, AST, bilirubin) markers are normal.  

*These lipid abnormalities merit discussion with a qualified clinician for a full risk assessment and possible treatment plan.*  

---

### INDIAN DIET PLAN  

#### 1. Foods to avoid / limit  
- **Refined grains & sugary foods** – white rice, maida (refined flour) rotis, sweets (jalebi, laddu, peda), sugary drinks, packaged snacks.  
- **Deep‑fried items** – samosas, pakor

In [5]:
def analyze_blood_work(blood_report: str):
    # Stage 1: Extract and classify results
    extraction_prompt = f"""
    Extract every test result from the blood report.

    For each result, provide:
    - Test name
    - Measured value
    - Status: HIGH, LOW, NORMAL, or UNDETERMINED
    - Reference range

    Use only the values and reference ranges provided.
    Do not invent missing information.

    Blood report:
    {blood_report}
    """

    stage1_response = llm.invoke(extraction_prompt)
    extracted_values = stage1_response.content

    # Stage 2: Generate summary and dietary suggestions
    diet_prompt = f"""
    Based on the following extracted blood results:

    1. Write a simple health summary in 4–5 lines.
    2. Provide an Indian diet plan with ONLY these sections:
       (1) Foods to avoid
       (2) Foods to eat more of

    Do not diagnose a disease, invent missing information,
    or recommend medication or supplements.
    Mention that significant abnormalities need clinical review.

    Blood work analysis:
    {extracted_values}
    """

    stage2_response = llm.invoke(diet_prompt)

    return {
        "extracted_values": extracted_values,
        "health_summary_and_diet": stage2_response.content
    }


# Execute the pipeline
result = analyze_blood_work(blood_report)

print("=== STAGE 1: EXTRACTED VALUES ===")
print(result["extracted_values"])

print("\n=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(result["health_summary_and_diet"])

=== STAGE 1: EXTRACTED VALUES ===
**Extracted Test Results**

| Test Name | Measured Value | Reference Range | Status |
|-----------|----------------|----------------|--------|
| Hemoglobin | 15.1 g/dL | 13.5 – 17.5 g/dL | NORMAL |
| Hematocrit | 44 % | 41 – 53 % | NORMAL |
| White Blood Cell (WBC) count | 6.8 ×10³/µL | 4.5 – 11.0 ×10³/µL | NORMAL |
| Platelets | 220 ×10³/µL | 150 – 400 ×10³/µL | NORMAL |
| Total Cholesterol | 238 mg/dL | \<200 mg/dL | HIGH |
| LDL Cholesterol | 162 mg/dL | \<100 mg/dL | HIGH |
| HDL Cholesterol | 36 mg/dL | \>40 mg/dL | LOW |
| Triglycerides | 188 mg/dL | \<150 mg/dL | HIGH |
| Glucose (Fasting) | 92 mg/dL | 70 – 99 mg/dL | NORMAL |
| HbA1c | 5.3 % | \<5.7 % | NORMAL |
| Creatinine | 1.0 mg/dL | 0.7 – 1.3 mg/dL | NORMAL |
| eGFR | 82 mL/min | \>60 mL/min | NORMAL |
| ALT (Alanine Aminotransferase) | 28 U/L | 7 – 40 U/L | NORMAL |
| AST (Aspartate Aminotransferase) | 25 U/L | 10 – 40 U/L | NORMAL |
| Bilirubin, Total | 0.8 mg/dL | 0.2 – 1.2 mg/dL | NOR